# 05 — Fusion multimodale et évaluation finale

**Objectif** : combiner clinique, image (DermLIP) et concepts, choisir **un** modèle par une règle fixée à l'avance, puis l'évaluer **une seule fois** sur l'ensemble de test.

| ID | Modèle | Entrées |
|---|---|---|
| E1a | Logistique | clinique |
| E3 | Logistique | image (plongement DermLIP, réduit par ACP) |
| E5 | Logistique | concepts |
| E6a | Fusion **précoce** (logistique) | image + clinique |
| E6b | Fusion **tardive** (stacking) | image + clinique |
| E7a | Fusion précoce | image + concepts + clinique |
| E7c | Fusion précoce, **sans plongement brut** | concepts + clinique (100 % nommé) |
| E7b | Fusion **à porte** (petit réseau) | image + concepts + clinique |
| E8 | Expérience secondaire : profil patient complet | sous-ensemble de 1 494 lignes |
| E9 | Optionnel (GPU) : ajustement fin de DermLIP | image + clinique |

**Protocole** : mêmes 5 plis groupés par patient ; tous les hyperparamètres (C, dimension de l'ACP) choisis à l'intérieur des plis (validation croisée imbriquée, groupée par patient).

**Règle de choix (fixée avant de voir les résultats)** : parmi les modèles de fusion, on prend la meilleure AUC moyenne ; si un modèle **plus simple** est à moins de 0,005 d'AUC, on prend le plus simple. Ordre de simplicité : E7c < E6a < E7a < E6b < E7b.

**Prérequis** : notebooks 03 et 04 exécutés (caches DermLIP et concepts). Aucun téléchargement dans ce notebook, sauf la section optionnelle E9.

## 0. Imports, données, fonctions et modèles communs

In [ ]:
from pathlib import Path
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, f1_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = ROOT / "zr7vgbcyr2-1" / "metadata.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
EXTERNAL_DIR = ROOT / "data" / "external" / "derm1m"
CONFIG_DIR = ROOT / "configs"
FIG_DIR = ROOT / "outputs" / "figures"
REPORT_DIR = ROOT / "outputs" / "reports"
CKPT_DIR = ROOT / "outputs" / "checkpoints"
for d in (PROCESSED_DIR, FIG_DIR, REPORT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# variables cliniques retenues (notebook 01) — le profil patient n'est utilisé que dans l'expérience E8
NUM_COLS = ["age"]
BOOL_COLS = ["itch", "grew", "hurt", "changed", "bleed", "elevation"]
CAT_COLS = ["region"]
FEATURES = NUM_COLS + BOOL_COLS + CAT_COLS
PROFILE_NUM = ["fitspatrick", "diameter_1", "diameter_2"]
PROFILE_BOOL = ["smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history"]
PROFILE_CAT = ["gender"]
PROFILE_COLS = PROFILE_NUM + PROFILE_BOOL + PROFILE_CAT

CANCER_CLASSES = ["BCC", "SCC", "MEL"]
DERMLIP_ID = "hf-hub:redlessone/DermLIP_ViT-B-16"

In [ ]:
splits = pd.read_csv(PROCESSED_DIR / "splits.csv")
meta = pd.read_csv(CSV_PATH, na_values=["UNK"])

df = splits.merge(meta[["img_id"] + FEATURES + PROFILE_COLS], on="img_id", how="left",
                  validate="one_to_one")
TO_FLOAT = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}
for c in BOOL_COLS + PROFILE_BOOL:                     # True / False / NaN -> 1.0 / 0.0 / NaN
    df[c] = df[c].map(TO_FLOAT)

dev = df[df["split"] == "train"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
y_dev, folds, groups_dev = dev["label"].to_numpy(), dev["fold"].to_numpy(), dev["patient_id"].to_numpy()
y_test, groups_test = test["label"].to_numpy(), test["patient_id"].to_numpy()

print(f"Développement : {len(dev)} images, {dev['patient_id'].nunique()} patients | "
      f"test : {len(test)} images (non utilisé avant l'évaluation finale)")
assert set(folds) == set(range(N_FOLDS))

In [ ]:
def metrics_at(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, proba),
        "Exactitude équilibrée": balanced_accuracy_score(y_true, y_pred),
        "Sensibilité": recall_score(y_true, y_pred, pos_label=1),
        "Spécificité": recall_score(y_true, y_pred, pos_label=0),
        "F1": f1_score(y_true, y_pred),
    }


def take_rows(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]


def fit_model(model, X, y, groups=None):
    """Entraîne un modèle ; transmet `groups` aux modèles qui choisissent leurs hyperparamètres par patient."""
    needs_groups = isinstance(model, GridSearchCV) or getattr(model, "uses_groups", False)
    if needs_groups:
        assert groups is not None, "ce modèle a besoin des identifiants patients (groups)"
        return model.fit(X, y, groups=groups)
    return model.fit(X, y)


def cross_validate(make_model, X, y, folds, groups=None, return_models=False):
    """Validation croisée sur les 5 plis groupés par patient : métriques par pli + prédictions hors pli."""
    oof, rows, models = np.zeros(len(y)), [], []
    for k in range(N_FOLDS):
        tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
        model = make_model()
        fit_model(model, take_rows(X, tr), y[tr], None if groups is None else groups[tr])
        oof[va] = model.predict_proba(take_rows(X, va))[:, 1]
        rows.append({"pli": k, **metrics_at(y[va], oof[va])})
        models.append(model)
    per_fold = pd.DataFrame(rows)
    return (per_fold, oof, models) if return_models else (per_fold, oof)


def summarize(name, per_fold):
    m = per_fold.drop(columns="pli")
    return {"Expérience": name, **{c: f"{m[c].mean():.3f} ± {m[c].std():.3f}" for c in m.columns},
            "AUC moyenne": m["AUC"].mean()}


def grouped_bootstrap(y, p, groups, p_ref=None, n_boot=2000, seed=SEED):
    """AUC et IC 95 % par bootstrap des patients ; avec `p_ref`, différence d'AUC appariée."""
    y, p, groups = np.asarray(y), np.asarray(p), np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    rng = np.random.default_rng(seed)
    aucs, diffs = [], []
    for _ in range(n_boot):
        idx = np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))])
        if y[idx].min() == y[idx].max():
            continue
        a = roc_auc_score(y[idx], p[idx])
        aucs.append(a)
        if p_ref is not None:
            diffs.append(a - roc_auc_score(y[idx], np.asarray(p_ref)[idx]))
    out = {"AUC": roc_auc_score(y, p), "IC95 bas": np.percentile(aucs, 2.5), "IC95 haut": np.percentile(aucs, 97.5)}
    if p_ref is not None:
        d = np.asarray(diffs)
        out.update({"ΔAUC": out["AUC"] - roc_auc_score(y, p_ref),
                    "ΔAUC IC95 bas": np.percentile(d, 2.5), "ΔAUC IC95 haut": np.percentile(d, 97.5),
                    "P(Δ ≤ 0)": float((d <= 0).mean())})
    return out


def make_clinical_preprocessor(num=NUM_COLS, bools=BOOL_COLS, cats=CAT_COLS):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("bool", SimpleImputer(strategy="most_frequent"), bools),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cats),
    ])


C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]


def make_probe(c_grid=C_GRID):
    """Sonde linéaire : standardisation + logistique, C choisi par validation croisée groupée (patients)."""
    pipe = Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
    return GridSearchCV(pipe, {"clf__C": c_grid}, cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1)


def load_features(name):
    """Charge un cache de caractéristiques et renvoie une fonction frame -> matrice alignée sur frame['img_id']."""
    feats = np.load(PROCESSED_DIR / f"features_{name}.npy")
    ids = pd.read_csv(PROCESSED_DIR / f"features_{name}_ids.csv")["img_id"]
    index = pd.Series(np.arange(len(ids)), index=ids.to_numpy())
    return lambda frame: feats[index.loc[frame["img_id"]].to_numpy()]


results, oof_preds = [], {}

In [ ]:
from functools import partial

import joblib
import torch
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit


def make_linear_fusion(clin=True, img=False, cpt=False, profile=False):
    """Fusion précoce : blocs (clinique / image / concepts) prétraités puis une régression logistique.
    C (et la réduction PCA de l'image) sont choisis par validation croisée groupée par patient."""
    blocks = []
    if clin:
        num, bools, cats = ((NUM_COLS + PROFILE_NUM, BOOL_COLS + PROFILE_BOOL, CAT_COLS + PROFILE_CAT)
                            if profile else (NUM_COLS, BOOL_COLS, CAT_COLS))
        blocks.append(("clin", make_clinical_preprocessor(num, bools, cats), num + bools + cats))
    if img:
        blocks.append(("img", Pipeline([("scale", StandardScaler()), ("pca", PCA(random_state=SEED))]), IMG_COLS))
    if cpt:
        blocks.append(("cpt", StandardScaler(), CONCEPT_COLS))
    pipe = Pipeline([("prep", ColumnTransformer(blocks)), ("scale", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
    grid = {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]}
    if img:
        grid["prep__img__pca__n_components"] = [16, 64, 128]
    return GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1)


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


class LateFusion(BaseEstimator, ClassifierMixin):
    """Fusion tardive (stacking) : un modèle par modalité, combinés par une logistique apprise
    sur leurs prédictions hors pli internes (groupées par patient)."""
    uses_groups = True

    def __init__(self, base_factories=(), n_inner=3):
        self.base_factories = base_factories
        self.n_inner = n_inner

    def fit(self, X, y, groups):
        y, groups = np.asarray(y), np.asarray(groups)
        Z = np.zeros((len(y), len(self.base_factories)))
        for tr, va in GroupKFold(n_splits=self.n_inner).split(X, y, groups):
            for j, (_, factory) in enumerate(self.base_factories):
                m = fit_model(factory(), take_rows(X, tr), y[tr], groups[tr])
                Z[va, j] = _logit(m.predict_proba(take_rows(X, va))[:, 1])
        self.meta_ = LogisticRegression(class_weight="balanced").fit(Z, y)
        self.bases_ = [fit_model(f(), X, y, groups) for _, f in self.base_factories]
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        Z = np.column_stack([_logit(m.predict_proba(X)[:, 1]) for m in self.bases_])
        return self.meta_.predict_proba(Z)


class _GatedNet(torch.nn.Module):
    def __init__(self, dims, hidden, dropout):
        super().__init__()
        self.encoders = torch.nn.ModuleList([
            torch.nn.Sequential(torch.nn.Linear(d, hidden), torch.nn.GELU(), torch.nn.Dropout(dropout)) for d in dims])
        self.gate = torch.nn.Linear(hidden * len(dims), len(dims))
        self.head = torch.nn.Sequential(torch.nn.LayerNorm(hidden), torch.nn.Linear(hidden, 1))

    def forward(self, xs):
        hs = [enc(x) for enc, x in zip(self.encoders, xs)]
        w = torch.softmax(self.gate(torch.cat(hs, dim=1)), dim=1)        # poids de chaque modalité
        fused = sum(w[:, i:i + 1] * h for i, h in enumerate(hs))
        return self.head(fused).squeeze(1), w


class GatedFusion(BaseEstimator, ClassifierMixin):
    """Fusion à porte : un petit réseau par modalité, une porte apprend le poids de chaque modalité
    pour chaque patient. Arrêt anticipé sur 15 % des patients du pli ; moyenne de plusieurs graines."""
    uses_groups = True
    MODALITIES = ("clinique", "image", "concepts")

    def __init__(self, hidden=32, dropout=0.3, lr=1e-3, weight_decay=1e-2, max_epochs=400,
                 patience=30, n_seeds=5, val_size=0.15, img_pca=64):
        self.hidden, self.dropout, self.lr, self.weight_decay = hidden, dropout, lr, weight_decay
        self.max_epochs, self.patience, self.n_seeds = max_epochs, patience, n_seeds
        self.val_size, self.img_pca = val_size, img_pca

    def _blocks(self):
        return [("clin", make_clinical_preprocessor(), FEATURES),
                ("img", Pipeline([("scale", StandardScaler()),
                                  ("pca", PCA(n_components=self.img_pca, random_state=SEED))]), IMG_COLS),
                ("cpt", StandardScaler(), CONCEPT_COLS)]

    def _transform(self, X):
        return [torch.tensor(np.asarray(p.transform(X[cols]), dtype=np.float32))
                for (_, _, cols), p in zip(self._blocks(), self.preps_)]

    def fit(self, X, y, groups):
        y = np.asarray(y, dtype=np.float32)
        tr, va = next(GroupShuffleSplit(n_splits=1, test_size=self.val_size, random_state=SEED)
                      .split(X, y, groups))
        Xtr, Xva = take_rows(X, tr), take_rows(X, va)
        self.preps_ = [p.fit(Xtr[cols]) for _, p, cols in self._blocks()]
        xtr, xva = self._transform(Xtr), self._transform(Xva)
        ytr, yva = torch.tensor(y[tr]), torch.tensor(y[va])
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - y[tr]).sum() / max(y[tr].sum(), 1.0)))
        self.nets_ = []
        for s in range(self.n_seeds):
            torch.manual_seed(SEED + s)
            net = _GatedNet([x.shape[1] for x in xtr], self.hidden, self.dropout)
            opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            best, best_state, wait = np.inf, None, 0
            for _ in range(self.max_epochs):
                net.train()
                opt.zero_grad()
                loss_fn(net(xtr)[0], ytr).backward()
                opt.step()
                net.eval()
                with torch.no_grad():
                    v = loss_fn(net(xva)[0], yva).item()
                if v < best - 1e-4:
                    best, wait = v, 0
                    best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
                else:
                    wait += 1
                    if wait >= self.patience:
                        break
            net.load_state_dict(best_state)
            net.eval()
            self.nets_.append(net)
        self.classes_ = np.array([0, 1])
        return self

    def _forward(self, X):
        xs = self._transform(X)
        with torch.no_grad():
            outs = [net(xs) for net in self.nets_]
        p = np.mean([torch.sigmoid(o[0]).numpy() for o in outs], axis=0)
        w = np.mean([o[1].numpy() for o in outs], axis=0)
        return p, w

    def predict_proba(self, X):
        p, _ = self._forward(X)
        return np.column_stack([1 - p, p])

    def modality_weights(self, X):
        return pd.DataFrame(self._forward(X)[1], columns=list(self.MODALITIES))


def expected_calibration_error(y, p, n_bins=10):
    y, p = np.asarray(y), np.asarray(p)
    idx = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    rows, ece = [], 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(y[m].mean() - p[m].mean())
            rows.append({"confiance moyenne": p[m].mean(), "fréquence observée": y[m].mean(), "n": int(m.sum())})
    return ece, pd.DataFrame(rows)

In [ ]:
take_img = load_features("dermlip_vitb16")
emb_bank = np.load(PROCESSED_DIR / "concept_text_embeddings.npz", allow_pickle=True)
selected = pd.read_csv(PROCESSED_DIR / "concepts_selected.csv")
CONCEPT_NAMES = selected.loc[selected["retenu"], "concept"].tolist()
_name_to_row = {n: i for i, n in enumerate(emb_bank["names"])}
CPT_EMB = emb_bank["concept_emb"][[_name_to_row[n] for n in CONCEPT_NAMES]]
REF_EMB = emb_bank["ref_emb"]

N_IMG = take_img(dev.head(1)).shape[1]
IMG_COLS = [f"img_{i}" for i in range(N_IMG)]
CONCEPT_COLS = [f"c_{n}" for n in CONCEPT_NAMES]


def concept_scores(emb):
    """Score de concept = similarité image/concept moins similarité image/référence neutre."""
    return emb @ CPT_EMB.T - (emb @ REF_EMB)[:, None]


def build_frame(frame, emb=None):
    """Table complète : variables cliniques + plongement image DermLIP + scores de concepts."""
    emb = take_img(frame) if emb is None else emb
    clin = frame[FEATURES + PROFILE_COLS].reset_index(drop=True)
    img = pd.DataFrame(emb, columns=IMG_COLS)
    cpt = pd.DataFrame(concept_scores(emb), columns=CONCEPT_COLS)
    return pd.concat([clin, img, cpt], axis=1)


print("Concepts retenus :", len(CONCEPT_NAMES), "| dimension image :", N_IMG)

## 1. Tables d'entrée

In [ ]:
X_dev = build_frame(dev)
X_test = build_frame(test)
print("Développement :", X_dev.shape, "| test :", X_test.shape)

overlap_path = PROCESSED_DIR / "derm1m_overlap_ids.csv"
overlap_ids = set(pd.read_csv(overlap_path)["img_id"]) if overlap_path.exists() else set()
print("Images signalées par le contrôle de chevauchement Derm1M :", len(overlap_ids))

## 2. Validation croisée de tous les modèles

In [ ]:
EXPERIMENTS = {
    "E1a Clinique": lambda: make_linear_fusion(clin=True),
    "E3 Image DermLIP": lambda: make_linear_fusion(clin=False, img=True),
    "E5 Concepts": lambda: make_linear_fusion(clin=False, cpt=True),
    "E6a Précoce image+clinique": lambda: make_linear_fusion(clin=True, img=True),
    "E6b Tardive image+clinique": lambda: LateFusion((          # partial (et non lambda) : modèle enregistrable
        ("clinique", partial(make_linear_fusion, clin=True)),
        ("image", partial(make_linear_fusion, clin=False, img=True)))),
    "E7a Précoce image+concepts+clinique": lambda: make_linear_fusion(clin=True, img=True, cpt=True),
    "E7c Concepts+clinique": lambda: make_linear_fusion(clin=True, cpt=True),
    "E7b À porte (réseau)": lambda: GatedFusion(),
}
SIMPLICITY = ["E7c Concepts+clinique", "E6a Précoce image+clinique", "E7a Précoce image+concepts+clinique",
              "E6b Tardive image+clinique", "E7b À porte (réseau)"]

per_fold_all = {}
for name, factory in EXPERIMENTS.items():
    t0 = time.time()
    per_fold, oof_preds[name] = cross_validate(factory, X_dev, y_dev, folds, groups=groups_dev)
    per_fold_all[name] = per_fold
    results.append(summarize(name, per_fold))
    print(f"{name:42s} AUC {per_fold['AUC'].mean():.3f} ± {per_fold['AUC'].std():.3f}   ({time.time() - t0:.0f} s)")

In [ ]:
table = pd.DataFrame(results).set_index("Expérience")
auc_mean = table["AUC moyenne"]
table.drop(columns="AUC moyenne").to_csv(REPORT_DIR / "05_validation_croisee.csv")
table.drop(columns="AUC moyenne").sort_values("AUC", ascending=False)

## 3. Choix du modèle (règle fixée à l'avance)

In [ ]:
best = max(SIMPLICITY, key=lambda n: auc_mean[n])
CHOSEN = next(n for n in SIMPLICITY if auc_mean[n] >= auc_mean[best] - 0.005)
print(f"Meilleure AUC : {best} ({auc_mean[best]:.3f})")
print(f"Modèle retenu : {CHOSEN} ({auc_mean[CHOSEN]:.3f})")

rows = []
for ref in ["E1a Clinique", "E3 Image DermLIP", "E5 Concepts", best]:
    if ref != CHOSEN:
        r = grouped_bootstrap(y_dev, oof_preds[CHOSEN], groups_dev, p_ref=oof_preds[ref])
        rows.append({"comparaison": f"{CHOSEN}  vs  {ref}", **{k: round(v, 3) for k, v in r.items()}})
comparisons = pd.DataFrame(rows)
comparisons.to_csv(REPORT_DIR / "05_comparaisons_bootstrap.csv", index=False)
comparisons

In [ ]:
fold_table = pd.DataFrame({n: per_fold_all[n]["AUC"].to_numpy() for n in EXPERIMENTS}, index=[f"pli {k}" for k in range(N_FOLDS)])
plt.figure(figsize=(9, 4.5))
for n in fold_table:
    plt.plot(fold_table.index, fold_table[n], marker="o", lw=2.5 if n == CHOSEN else 1, label=n)
plt.ylabel("AUC du pli")
plt.title("AUC par pli (les plis sont appariés : mêmes patients pour tous les modèles)")
plt.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_auc_par_pli.png", dpi=150)
plt.show()

### Poids des modalités (modèle à porte)

La porte du modèle E7b attribue à chaque patient un poids par modalité : c'est une première mesure de **ce sur quoi le modèle s'appuie**.

In [ ]:
gated = fit_model(GatedFusion(), X_dev, y_dev, groups_dev)
w = gated.modality_weights(X_dev)
print("Poids moyens des modalités :", w.mean().round(3).to_dict())
w.groupby(dev["diagnostic"].to_numpy()).mean().round(3)

## 4. E8 — Expérience secondaire : profil patient complet

Sur les seules lignes où le profil patient est renseigné (pas de raccourci possible : il n'y a plus de valeurs manquantes à exploiter), on mesure ce qu'apportent le diamètre, le phototype, les antécédents… **Ces chiffres ne sont pas comparables aux précédents** : ce sous-ensemble contient environ 73 % de cancers.

In [ ]:
complete = ~dev[PROFILE_COLS].isna().any(axis=1).to_numpy()
Xs, ys, fs, gs = X_dev[complete].reset_index(drop=True), y_dev[complete], folds[complete], groups_dev[complete]
print(f"Sous-ensemble : {complete.sum()} images, {len(set(gs))} patients, {ys.mean() * 100:.1f} % de cancers")

E8 = {"E8a Clinique (sous-ensemble)": lambda: make_linear_fusion(clin=True),
      "E8b Clinique + profil": lambda: make_linear_fusion(clin=True, profile=True),
      "E8c Image+concepts+clinique + profil": lambda: make_linear_fusion(clin=True, img=True, cpt=True, profile=True)}
e8_rows, e8_oof = [], {}
for name, factory in E8.items():
    per_fold, e8_oof[name] = cross_validate(factory, Xs, ys, fs, groups=gs)
    e8_rows.append(summarize(name, per_fold))
e8 = pd.DataFrame(e8_rows).set_index("Expérience").drop(columns="AUC moyenne")
e8.to_csv(REPORT_DIR / "05_E8_profil_complet.csv")
r = grouped_bootstrap(ys, e8_oof["E8b Clinique + profil"], gs, p_ref=e8_oof["E8a Clinique (sous-ensemble)"])
print("Apport du profil (E8b − E8a) :", {k: round(v, 3) for k, v in r.items()})
e8

## 5. E9 — Optionnel : ajustement fin de DermLIP (GPU)

Exécutée **seulement si un GPU est disponible** (par exemple sur Colab ou Kaggle). Les images sont d'abord **redimensionnées une seule fois** (256 px, en parallèle sur tous les cœurs) et mises en cache : sans ce cache, le décodage des grandes images PNG à chaque époque sature le processeur et le GPU reste inactif (≈ 5 h au lieu de quelques minutes). Les 2 derniers blocs du ViT et la projection sont dégelés, une petite tête combine le plongement image et les variables cliniques. Nombre d'époques fixé à l'avance (pas de réglage sur la validation). Résultat **exploratoire** : il n'entre pas dans la règle de choix, mais indique le potentiel d'un entraînement complet.

In [ ]:
RUN_FINETUNE = torch.cuda.is_available()
FT_EPOCHS, FT_BATCH, FT_UNFREEZE = 12, 32, 2

if not RUN_FINETUNE:
    print("Pas de GPU : section E9 ignorée.")
else:
    import open_clip
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms as T

    import os
    import multiprocessing as mp

    # « fork » : les processus héritent des fonctions définies dans le notebook (Linux : Kaggle, Colab)
    MP_CTX = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None

    device = "cuda"
    CACHE_DIR = ROOT / "data" / "cache_256"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_SIDE = 256

    def cache_path(img_id):
        return CACHE_DIR / (Path(img_id).stem + ".jpg")

    def resize_one(job):
        src, dst = job
        if not dst.exists():
            with Image.open(src) as im:
                im = im.convert("RGB")
                w, h = im.size
                s = CACHE_SIDE / min(w, h)
                im.resize((max(CACHE_SIDE, round(w * s)), max(CACHE_SIDE, round(h * s))),
                          Image.Resampling.BICUBIC).save(dst, quality=95)
        return 1

    t0 = time.time()
    jobs = [(ROOT / p, cache_path(i)) for i, p in zip(dev["img_id"], dev["img_path"])]
    with (MP_CTX or mp).Pool(os.cpu_count()) as pool:
        done = sum(pool.imap_unordered(resize_one, jobs, chunksize=16))
    print(f"Cache d'images prêt : {done} images en {time.time() - t0:.0f} s ({CACHE_DIR})")

    _, _, eval_tf = open_clip.create_model_and_transforms(DERMLIP_ID)
    normalize = next(t for t in eval_tf.transforms if isinstance(t, T.Normalize))
    train_tf = T.Compose([
        T.RandomResizedCrop(224, scale=(0.7, 1.0), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
        T.ColorJitter(0.1, 0.1, 0.1, 0.02), T.ToTensor(), normalize])

    class LesionDataset(Dataset):
        def __init__(self, frame, clin, tf):
            self.paths = [cache_path(i) for i in frame["img_id"]]     # images redimensionnées (cache)
            self.clin = torch.tensor(np.asarray(clin, dtype=np.float32))
            self.y = torch.tensor(frame["label"].to_numpy(), dtype=torch.float32)
            self.tf = tf

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, i):
            with Image.open(self.paths[i]) as im:
                return self.tf(im.convert("RGB")), self.clin[i], self.y[i]

    class FineTuneNet(torch.nn.Module):
        def __init__(self, clip_model, n_clin):
            super().__init__()
            self.visual = clip_model.visual
            self.head = torch.nn.Sequential(torch.nn.Linear(self.visual.output_dim + n_clin, 64), torch.nn.GELU(),
                                            torch.nn.Dropout(0.3), torch.nn.Linear(64, 1))

        def forward(self, x, clin):
            emb = torch.nn.functional.normalize(self.visual(x), dim=-1)
            return self.head(torch.cat([emb, clin], dim=1)).squeeze(1)

    ft_oof, ft_rows = np.zeros(len(dev)), []
    for k in range(N_FOLDS):
        t0 = time.time()
        tr, va = folds != k, folds == k
        prep = make_clinical_preprocessor().fit(dev.loc[tr, FEATURES])
        clip_k, _, _ = open_clip.create_model_and_transforms(DERMLIP_ID)
        net = FineTuneNet(clip_k, prep.transform(dev.loc[tr, FEATURES]).shape[1]).to(device)
        for p in net.visual.parameters():
            p.requires_grad = False
        trainable = list(net.visual.transformer.resblocks[-FT_UNFREEZE:].parameters()) + list(net.visual.ln_post.parameters())
        if isinstance(net.visual.proj, torch.nn.Parameter):
            trainable.append(net.visual.proj)
        for p in trainable:
            p.requires_grad = True
        opt = torch.optim.AdamW([{"params": trainable, "lr": 1e-5}, {"params": net.head.parameters(), "lr": 1e-3}],
                                weight_decay=0.05)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
        ytr = dev.loc[tr, "label"].to_numpy()
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - ytr).sum() / ytr.sum(), device=device))
        scaler = torch.amp.GradScaler("cuda")
        train_dl = DataLoader(LesionDataset(dev[tr], prep.transform(dev.loc[tr, FEATURES]), train_tf),
                              batch_size=FT_BATCH, shuffle=True, num_workers=os.cpu_count(), pin_memory=True,
                              drop_last=True, persistent_workers=True, multiprocessing_context=MP_CTX)
        val_dl = DataLoader(LesionDataset(dev[va], prep.transform(dev.loc[va, FEATURES]), eval_tf),
                            batch_size=64, shuffle=False, num_workers=os.cpu_count(), pin_memory=True,
                            multiprocessing_context=MP_CTX)
        for epoch in range(FT_EPOCHS):
            net.train()
            for x, c, y in train_dl:
                opt.zero_grad()
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = loss_fn(net(x.to(device), c.to(device)), y.to(device))
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            sched.step()
            if k == 0 and epoch == 0:
                print(f"  pli 0, 1re époque : {time.time() - t0:.0f} s (estimation du temps total ≈ {(time.time() - t0) * FT_EPOCHS * N_FOLDS / 60:.0f} min)")
        net.eval()
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            ft_oof[va] = np.concatenate([torch.sigmoid(net(x.to(device), c.to(device))).float().cpu().numpy()
                                         for x, c, _ in val_dl])
        ft_rows.append({"pli": k, **metrics_at(y_dev[va], ft_oof[va])})
        print(f"pli {k} : AUC {ft_rows[-1]['AUC']:.3f} ({time.time() - t0:.0f} s)")
        del net, clip_k
        torch.cuda.empty_cache()

    oof_preds["E9 Ajustement fin (GPU)"] = ft_oof
    results.append(summarize("E9 Ajustement fin (GPU)", pd.DataFrame(ft_rows)))
    r = grouped_bootstrap(y_dev, ft_oof, groups_dev, p_ref=oof_preds[CHOSEN])
    print("E9 vs modèle retenu :", {k: round(v, 3) for k, v in r.items()})

## 6. Évaluation finale sur le test

> ⚠ **Le test n'est utilisé qu'ici, une seule fois**, avec le modèle choisi à la section 3. Ne pas revenir modifier le modèle après avoir vu ces résultats : sinon le test devient un jeu de validation et les chiffres sont optimistes.

Les seuils de décision sont fixés **sur les prédictions hors pli du développement** :
- **Youden** : maximise sensibilité + spécificité ;
- **Triage** : premier seuil atteignant une sensibilité ≥ 90 % (manquer un cancer coûte plus cher qu'une consultation de plus) — **point de fonctionnement principal**.

In [ ]:
final_model = fit_model(EXPERIMENTS[CHOSEN](), X_dev, y_dev, groups_dev)
clinical_model = fit_model(EXPERIMENTS["E1a Clinique"](), X_dev, y_dev, groups_dev)

fpr, tpr, thr = roc_curve(y_dev, oof_preds[CHOSEN])
THRESHOLDS = {"0,5": 0.5,
              "Youden (dev)": float(thr[np.argmax(tpr - fpr)]),
              "Triage sensibilité ≥ 90 % (dev)": float(thr[np.argmax(tpr >= 0.90)])}
print("Seuils fixés sur le développement :", {k: round(v, 3) for k, v in THRESHOLDS.items()})

p_test = final_model.predict_proba(X_test)[:, 1]
p_test_clin = clinical_model.predict_proba(X_test)[:, 1]
boot = grouped_bootstrap(y_test, p_test, groups_test, p_ref=p_test_clin)
print(f"\nTEST — {CHOSEN} : AUC {boot['AUC']:.3f} [IC95 {boot['IC95 bas']:.3f} – {boot['IC95 haut']:.3f}]")
print(f"Gain sur la clinique seule : ΔAUC {boot['ΔAUC']:+.3f} [IC95 {boot['ΔAUC IC95 bas']:+.3f} – {boot['ΔAUC IC95 haut']:+.3f}]")

operating = pd.DataFrame({k: metrics_at(y_test, p_test, t) for k, t in THRESHOLDS.items()}).T.round(3)
ece, reliability = expected_calibration_error(y_test, p_test)
print(f"Erreur de calibration attendue (ECE) : {ece:.3f}")
operating

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for name, p in [(CHOSEN, p_test), ("E1a Clinique", p_test_clin)]:
    f, t, _ = roc_curve(y_test, p)
    axes[0].plot(f, t, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set(xlabel="1 − spécificité", ylabel="Sensibilité", title="ROC sur le test")
axes[0].legend(fontsize=8)

axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].plot(reliability["confiance moyenne"], reliability["fréquence observée"], marker="o")
axes[1].set(xlabel="probabilité prédite", ylabel="fréquence observée de cancer",
            title=f"Calibration sur le test (ECE {ece:.3f})")

t_main = THRESHOLDS["Triage sensibilité ≥ 90 % (dev)"]
detect = test.assign(pred=(p_test >= t_main).astype(int)).groupby("diagnostic")["pred"].mean().reindex(
    ["BCC", "SCC", "MEL", "ACK", "NEV", "SEK"])
axes[2].bar(detect.index, detect * 100, color=["#C44E52"] * 3 + ["#55A868"] * 3)
axes[2].set(ylabel="% classés « cancer »", title="Classés « cancer » par diagnostic (seuil triage)")
for i, v in enumerate(detect * 100):
    axes[2].text(i, v + 1, f"{v:.0f} %", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_evaluation_test.png", dpi=150)
plt.show()

pd.crosstab(pd.Series(y_test, name="vrai"), pd.Series((p_test >= t_main).astype(int), name="prédit (triage)"))

In [ ]:
if overlap_ids:
    keep = ~test["img_id"].isin(overlap_ids).to_numpy()
    r = grouped_bootstrap(y_test[keep], p_test[keep], groups_test[keep])
    print(f"Test sans les {(~keep).sum()} images signalées par le contrôle Derm1M : "
          f"AUC {r['AUC']:.3f} [IC95 {r['IC95 bas']:.3f} – {r['IC95 haut']:.3f}]")
else:
    print("Aucune image signalée par le contrôle de chevauchement (ou contrôle non vérifié : voir notebook 03).")

## 7. Sauvegarde du modèle final

In [ ]:
reference = {c: float(X_dev[c].median()) for c in NUM_COLS}                       # valeur « typique »
reference.update({c: X_dev[c].mode().iloc[0] for c in BOOL_COLS + CAT_COLS})        # utilisée par les explications
bundle = {
    "nom": CHOSEN,
    "modele": final_model,
    "modele_clinique": clinical_model,
    "seuils": THRESHOLDS,
    "seuil_principal": "Triage sensibilité ≥ 90 % (dev)",
    "dermlip_id": DERMLIP_ID,
    "concepts": CONCEPT_NAMES,
    "reference_clinique": reference,
    "reference_concepts": X_dev[CONCEPT_COLS].mean().to_dict(),
    "reference_image": X_dev[IMG_COLS].mean().to_numpy(),
    "date": time.strftime("%Y-%m-%d %H:%M"),
}
joblib.dump(bundle, CKPT_DIR / "final_model.joblib")

test_out = test[["img_id", "patient_id", "diagnostic", "label"]].assign(proba=p_test, proba_clinique=p_test_clin)
test_out.to_csv(REPORT_DIR / "05_predictions_test.csv", index=False)
summary = {"modele_retenu": CHOSEN, "auc_validation_croisee": round(float(auc_mean[CHOSEN]), 4),
           "test": {k: round(float(v), 4) for k, v in boot.items()}, "ece_test": round(float(ece), 4),
           "seuils": {k: round(v, 4) for k, v in THRESHOLDS.items()},
           "points_de_fonctionnement_test": operating.to_dict(orient="index")}
(REPORT_DIR / "05_synthese.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf8")
print("Enregistrés : final_model.joblib, 05_predictions_test.csv, 05_synthese.json")

## Conclusion (à compléter après exécution)

- **Modèle retenu** et justification par la règle fixée à l'avance.
- **Apport de la fusion** : ΔAUC du modèle retenu face à la clinique seule et à l'image seule (validation croisée et test, avec IC 95 %).
- **Test** : AUC [IC 95 %], sensibilité et spécificité au seuil de triage, calibration (ECE), taux de détection par diagnostic — en particulier le **mélanome** (peu de cas : interpréter avec prudence).
- **E8** : apport du profil patient quand il est disponible.
- **E9** (si GPU) : potentiel de l'ajustement fin.

**Étape suivante** : `06_xai_evaluation.ipynb`.